In [1]:
import numpy as np

from numba import njit

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import fastplotlib as fpl

import optuna

from dysts.maps import Henon

import time

Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),Apple M4,IntegratedGPU,Metal,


To silence this warning, use a fully namespaced name.


# Init

## Init Reservoir

In [2]:
steps = 20000

tau_steps = 1

transient_steps_chaos = int(steps * 0.1)
transient_steps_reservoir = int(steps * 0.1)

total_steps = steps + transient_steps_chaos + transient_steps_reservoir + tau_steps
total_steps_after_chaos = steps + transient_steps_reservoir + tau_steps

test_size = 0.2
test_steps = int(steps * test_size)

t = np.arange(0, total_steps)

In [3]:
henon_model = Henon()
henon_dataset = henon_model.make_trajectory(total_steps)
henon_dataset = henon_dataset[transient_steps_chaos:]

In [4]:
henon_scaler = StandardScaler()
henon_train_scaled = henon_scaler.fit_transform(henon_dataset[:-test_steps])
henon_test_scaled = henon_scaler.transform(henon_dataset[-test_steps:])
henon_scaled = np.concatenate((henon_train_scaled, henon_test_scaled), axis=0)

## Init Funcs

In [5]:
def scatter_plot(data_list, names=["Test", "Pred"], colors=["white", "magenta"]):
    fig = go.Figure()

    for i, data in enumerate(data_list):
        if data.ndim == 1:
            fig.add_trace(
                go.Scatter(
                    x=np.arange(len(data)),
                    y=data,
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        elif data.ndim == 2:
            fig.add_trace(
                go.Scatter(
                    x=data[:, 0],
                    y=data[:, 1],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )
        else:
            fig.add_trace(
                go.Scatter3d(
                    x=data[:, 0],
                    y=data[:, 1],
                    z=data[:, 2],
                    mode="markers",
                    name=names[i],
                    marker=dict(color=colors[i % len(colors)], size=1),
                )
            )

    fig.update_layout(template="plotly_dark", title="Attractor Comparison")

    return fig

In [6]:
def r_2_plots_grid(actual_list, predicted_list, titles):
    actual_list = actual_list.reshape(-1, 1) if actual_list.ndim == 1 else actual_list
    predicted_list = predicted_list.reshape(-1, 1) if predicted_list.ndim == 1 else predicted_list

    fig = make_subplots(rows=1, cols=actual_list.shape[1], subplot_titles=titles)

    for i in range(actual_list.shape[1]):
        actual = actual_list[:, i]
        predicted = predicted_list[:, i]
        r_2 = r2_score(actual, predicted)
        col = i + 1

        fig.add_trace(
            go.Scatter(
                x=actual,
                y=predicted,
                mode="markers",
                name="Data",
                marker=dict(color="rgba(50, 50, 200, 0.5)", size=5),
            ),
            row=1,
            col=col,
        )

        min_val, max_val = min(actual.min(), predicted.min()), max(
            actual.max(), predicted.max()
        )
        fig.add_trace(
            go.Scatter(
                x=[min_val, max_val],
                y=[min_val, max_val],
                mode="lines",
                name="Ideal",
                line=dict(color="firebrick", dash="dash"),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="Actual", row=1, col=col)
        fig.update_yaxes(title_text=f"Predicted (R²: {r_2:.4f})", row=1, col=col)

    fig.update_layout(showlegend=False, height=500, width=1000)
    return fig

In [7]:
def plot_grid(nodes_pos):
    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=nodes_pos[:, 0],
            y=nodes_pos[:, 1],
            mode="markers",
            marker=dict(size=8, color="teal"),
        )
    )

    fig.update_layout(
        title="Hexagonal Lattice Distribution",
        xaxis=dict(title="X Position", scaleanchor="y", scaleratio=1),
        yaxis=dict(title="Y Position"),
        plot_bgcolor="white",
        width=700,
        height=700,
    )

    return fig


In [8]:
def weight_plot(weights, dims):
    dim_names = ["x", "y", "z"][:dims]
    num_features_per_type = len(weights) // 2
    num_nodes = num_features_per_type // dims

    labels = []
    for n in range(num_nodes):
        for d in range(dims):
            labels.append(f"Node {n + 1} {dim_names[d]} Pos")
            labels.append(f"Node {n + 1} {dim_names[d]} Vel")

    weights_flat = weights.flatten()
    pos_part = weights_flat[:num_features_per_type]
    vel_part = weights_flat[num_features_per_type:]

    combined_weights = np.empty_like(weights_flat)
    combined_weights[0::2] = pos_part
    combined_weights[1::2] = vel_part

    fig = go.Figure(
        data=[
            go.Bar(
                x=labels,
                y=combined_weights,
                marker_color=np.where(combined_weights >= 0, "royalblue", "firebrick"),
            )
        ]
    )

    fig.update_layout(
        title="Reservoir Node Contribution (Feature Weights)",
        xaxis_title="Node State & Dimension",
        yaxis_title="Weight Value",
        template="plotly_white",
    )

    return fig

In [60]:
def spring_animation(
    disp,
    nodes_pos,
    connections_list,
    size=10,
    external=False,
    is_3d=False,
    frames_moved=5,
    max_frames=2000,
    animate=True,
    highlighted_nodes=None,
    wall_nodes=None,
    vel=None,
    steps_jump=1,
):
    disp = disp[::steps_jump]

    steps = disp.shape[0]
    num_nodes = nodes_pos.shape[0]
    dims = nodes_pos.shape[1]

    fig = (
        fpl.Figure(canvas="glfw" if external else "jupyter")
        if not is_3d
        else fpl.Figure(
            cameras="3d",
            controller_types="orbit",
            canvas="glfw" if external else "jupyter",
        )
    )

    node_colors = np.array(["magenta"] * num_nodes)
    if highlighted_nodes is not None:
        node_colors[highlighted_nodes] = "lime"
    if wall_nodes is not None and wall_nodes[0] != -1:
        node_colors[wall_nodes] = "blue"

    disp_reshaped = disp.reshape(steps, num_nodes, dims)
    disp_3d = np.pad(disp_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")
    nodes_pos_3d = np.pad(nodes_pos, ((0, 0), (0, 3 - dims)), mode="constant")
    coords = nodes_pos_3d + disp_3d[0]

    dots = fig[0, 0].add_scatter(
        data=coords.astype(np.float32), sizes=size, colors=node_colors
    )
    lines = [
        fig[0, 0].add_line(
            data=np.vstack([coords[int(row[0])], coords[int(row[1])]]).astype(
                np.float32
            ),
            thickness=2,
            colors="cyan",
        )
        for row in connections_list
    ]

    if vel is not None:
        vel = vel[::steps_jump]
        vel_reshaped = vel.reshape(steps, num_nodes, dims)
        vel_3d = np.pad(vel_reshaped, ((0, 0), (0, 0), (0, 3 - dims)), mode="constant")

        vel_lines = [
            fig[0, 0].add_line(
                data=np.vstack([coords[i], coords[i] + vel_3d[0, i]]).astype(np.float32),
                thickness=1.5,
                colors="yellow",
            )
            for i in range(num_nodes)
        ]

    frame_tracker = 0
    def update_springs(canvas):
        nonlocal frame_tracker
        frame_tracker = (frame_tracker + frames_moved) % steps
        if frame_tracker >= max_frames:
            canvas.clear_animations()

        coords = nodes_pos_3d + disp_3d[frame_tracker]

        dots.data = coords.astype(np.float32)

        for row, l in zip(connections_list, lines):
            src, dst = int(row[0]), int(row[1])
            l.data = np.vstack([coords[src], coords[dst]]).astype(np.float32)

        if vel is not None:
            for i in range(num_nodes):
                vel_lines[i].data = np.vstack(
                    [coords[i], coords[i] + vel_3d[frame_tracker, i]]
                ).astype(np.float32)

    if animate:
        fig.add_animations(update_springs)
    return fig

## Calc Init

In [10]:
@njit(cache=True)
def get_spring_forces(connections_list, disp, initial_pos, rest_lens, k_vals, num_nodes, dims):
    forces = np.zeros((num_nodes, dims))
    disp_reshaped = disp.reshape(num_nodes, dims)

    for i in range(len(connections_list)):
        idx_a = connections_list[i, 0]
        idx_b = connections_list[i, 1]

        delta = np.zeros(dims)
        dist_sq = 0.0
        for j in range(dims):
            pos_a = initial_pos[idx_a, j] + disp_reshaped[idx_a, j]
            pos_b = initial_pos[idx_b, j] + disp_reshaped[idx_b, j]
            delta[j] = pos_b - pos_a
            dist_sq += delta[j] ** 2

        dist = np.sqrt(dist_sq)

        mag = k_vals[i] * (dist - rest_lens[i])

        for j in range(dims):
            f_component = mag * (delta[j] / dist)
            forces[idx_a, j] += f_component
            forces[idx_b, j] -= f_component

    return forces.reshape(-1)

In [ ]:
@njit(cache=True)
def run_simulation(
    steps,
    dt,
    m_inv_diag,
    c_diag,
    U,
    initial_pos,
    connections_list,
    k_vals,
    rest_lens,
    wall_nodes=[-1],
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))
    acc = np.zeros(matrix_size)

    mask = np.ones(matrix_size)
    if wall_nodes[0] != -1:
        for wall in wall_nodes:
            idx = wall * dims
            mask[idx : idx + dims] = 0

    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + 0.5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

In [12]:
def mu(val, sigma):
    return np.log(val) - (sigma**2 / 2)

# Basic Spring

In [44]:
N = 12

x = np.arange(N)
nodes_pos = x.reshape(-1, 1)

In [45]:
rng_val = 42
sigma = 0

dt = 0.01
input_force = 3
m_val = 0.03
c_val = 0.5
k_val = 3
target_node_count = 3

In [46]:
rng = np.random.default_rng(rng_val)

N_step = int(N / (target_node_count + 1))
target_nodes = np.arange(1, (target_node_count + 1)) * N_step

wall_nodes = np.array([0, N - 1])

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

m_nodes = rng.lognormal(mean=mu(m_val, sigma), sigma=sigma, size=num_nodes)
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / np.repeat(m_nodes, dims)

c_nodes = rng.lognormal(mean=mu(c_val, sigma), sigma=sigma, size=num_nodes)
c_diag = np.repeat(c_nodes, dims)

In [47]:
rng = np.random.default_rng(rng_val)

node_ids = np.arange(x.size)
src_nodes = node_ids[:-1]
dst_nodes = node_ids[1:]
connections_list = np.column_stack((src_nodes, dst_nodes))

k_vals = rng.lognormal(mean=mu(k_val, sigma), sigma=sigma, size=num_nodes)

init_vecs = nodes_pos[connections_list[:, 0]] - nodes_pos[connections_list[:, 1]]
rest_lens = np.sqrt(np.sum(init_vecs**2, axis=1))

In [48]:
total_steps_with_free = steps + transient_steps_reservoir + tau_steps

U = np.zeros((total_steps_with_free, matrix_size))
for i, node_index in enumerate(target_nodes):
    U[:, node_index * dims] = henon_scaled[:, i % henon_scaled.shape[1]]

In [49]:
displacement, velocity = run_simulation(
    steps=total_steps_with_free,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    U=U * input_force,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    wall_nodes=wall_nodes,
)

In [52]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=target_nodes,
    external=True,
    steps_jump=1,
).show()

In [35]:
X = np.column_stack((displacement, velocity))

X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps:]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test = henon_scaler.inverse_transform(Y_test)

In [36]:
r_2 = r2_score(Y_test, Y_pred)
mse = root_mean_squared_error(Y_test, Y_pred)

print(r_2, mse)

0.5797305997901391 0.34724523972824206


# Pendulum Sim

In [ ]:
def pendulum_sim(
    steps,
    dt,
    m_inv_diag,
    c_diag,
    gravity,
    initial_pos,
    connections_list,
    k_vals,
    rest_lens,
    pivot_node=0,
    U=None,  # Optional custom external force matrix of shape (steps, matrix_size)
):
    num_nodes = initial_pos.shape[0]
    dims = initial_pos.shape[1]
    matrix_size = num_nodes * dims

    disp = np.zeros((steps, matrix_size))
    v = np.zeros((steps, matrix_size))

    # 1. Setup Mask: Fix the pivot node so it cannot move
    mask = np.ones(matrix_size)
    pivot_idx = pivot_node * dims
    mask[pivot_idx : pivot_idx + dims] = 0

    # 2. Setup or Validate External Force U
    masses = 1.0 / m_inv_diag
    vertical_axis = 1 if dims > 1 else 0

    if U is None:
        # Default behavior: Just gravity acting downward on non-pivot nodes
        U = np.zeros((steps, matrix_size))
        for i in range(steps):
            for n in range(num_nodes):
                if n == pivot_node:
                    continue
                U[i, n * dims + vertical_axis] = (
                    -masses[n * dims + vertical_axis] * gravity
                )
    else:
        # If U is provided, ensure gravity is either already built into U
        # or you can choose to add gravity automatically alongside your custom U:
        gravity_matrix = np.zeros((steps, matrix_size))
        for i in range(steps):
            for n in range(num_nodes):
                if n == pivot_node:
                    continue
                gravity_matrix[i, n * dims + vertical_axis] = (
                    -masses[n * dims + vertical_axis] * gravity
                )
        U = U + gravity_matrix  # Combines gravity with your custom input forces

    # Initial spring forces
    F_spring = get_spring_forces(
        connections_list, disp[0], initial_pos, rest_lens, k_vals, num_nodes, dims
    )

    # Velocity Verlet Integration Loop
    for i in range(1, steps):
        acc = m_inv_diag * (F_spring - c_diag * v[i - 1] + U[i - 1])
        acc *= mask

        disp[i] = disp[i - 1] + v[i - 1] * dt + acc * 0.5 * dt**2

        F_spring = get_spring_forces(
            connections_list, disp[i], initial_pos, rest_lens, k_vals, num_nodes, dims
        )

        acc_next = m_inv_diag * (F_spring - c_diag * (v[i - 1] + 0.5 * acc * dt) + U[i])
        acc_next *= mask

        v[i] = v[i - 1] + 0.5 * (acc + acc_next) * dt

    return disp, v

In [86]:
# --- 1. Setup Simulation Parameters ---
steps = 1000
dt = 0.01

# Setup 2 nodes: Node 0 (Pivot at origin), Node 1 (Bob hanging at x=1, y=0)
nodes_pos = np.array([[0.0, 0.0], [1.0, 0.0]])  # 2D coordinates

num_nodes = nodes_pos.shape[0]
dims = nodes_pos.shape[1]
matrix_size = num_nodes * dims

# Masses and inverse masses
masses = np.array([1.0, 1.0])  # Pivot mass doesn't matter since it's locked
m_inv_diag = np.repeat(1.0 / masses, dims)

# Damping (air resistance)
c_diag = np.repeat(0.1, matrix_size)

# Connect Node 0 to Node 1 with a stiff spring (acting as a rigid rod)
connections_list = np.array([[0, 1]])
k_vals = np.array([5000.0])  # High stiffness for rigidity
rest_lens = np.array([1.0])  # Length of the pendulum

# --- 2. Generate Random External Inputs (U) ---
# Shape must match (steps, matrix_size)
U_random = np.zeros((steps, matrix_size))

# Target the pendulum bob (Node 1) for external driving forces (RC input style)
target_node = 1
target_idx = target_node * dims

# Create random input forces for each time step (e.g., random pushing along X and Y)
# Scale factor controls how hard the random forces push
force_scale = 5.0
for i in range(steps):
    # Random force vector for dimensions (X, Y)
    rand_force = np.random.uniform(-1.0, 1.0, dims) * force_scale
    U_random[i, target_idx : target_idx + dims] = rand_force

# --- 3. Run Simulation with External Forces ---
disp, v = run_pendulum_simulation(
    steps=steps,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    gravity=9.81,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    pivot_node=0,
    U=U_random,  # Pass the random external force matrix here
)

In [83]:
N = 5

num_nodes = N + 1  # 1 pivot node + N bob nodes
dims = 2
matrix_size = num_nodes * dims

# 1. Initial Positions: Stack them in a straight line along the x-axis
# Node 0 is at (0,0), Node 1 is at (1,0), Node 2 is at (2,0), etc.
nodes_pos = np.zeros((num_nodes, dims))
for i in range(num_nodes):
    nodes_pos[i, 0] = float(i)  # Spread them out horizontally by 1 unit each
    nodes_pos[i, 1] = 0.0

# 2. Masses and inverse masses (Pivot index 0 doesn't matter, bobs set to 1.0)
masses = np.ones(num_nodes)
m_inv_diag = np.repeat(1.0 / masses, dims)

# 3. Damping
c_diag = np.repeat(0.1, matrix_size)

# 4. Connections: Chain them together sequentially (0->1, 1->2, 2->3, ..., N-1->N)
connections_list = np.array([[i, i + 1] for i in range(N)])

# 5. Spring stiffness and rest lengths for each connection
k_vals = np.full(N, 5000.0)  # High rigidity for all rods
rest_lens = np.ones(N)  # Each rod has a length of 1.0

# Run simulation
disp, v = run_pendulum_simulation(
    steps=1000,
    dt=0.01,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    gravity=9.81,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    pivot_node=0,
)

In [87]:
spring_animation(
    disp=disp,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    external=True,
    steps_jump=1,
).show()

# More

In [ ]:
import numpy as np

# from sklearn.linear_model import RidgeCV
# from sklearn.metrics import r2_score, root_mean_squared_error

# --- 1. Reservoir & Simulation Configuration ---
rng_val = 42
sigma = 0
dt = 0.01
input_force = 3.0
m_val = 0.1
c_val = 0.1
k_val = 5000.0  # High stiffness for rigid rods

rng = np.random.default_rng(rng_val)
total_steps_with_free = (
    steps + transient_steps_reservoir + tau_steps
)  # Assumes steps, etc., are defined

# --- 2. Define Node Layout: Cart + Double Pendulum ---
# Node 0: The Cart (slides horizontally along X-axis)
# Node 1: First pendulum bob
# Node 2: Second pendulum bob (tip)
# Plus extra nodes if you want to extend it into a larger chain/reservoir network!
num_nodes = 3
dims = 2  # 2D coordinates (X, Y)
matrix_size = num_nodes * dims

nodes_pos = np.array(
    [
        [0.0, 0.0],  # Node 0: Cart (at origin)
        [0.0, -1.0],  # Node 1: First bob hanging down
        [0.0, -2.0],  # Node 2: Second bob hanging down
    ]
)

# --- 3. Connections (Cart -> Rod 1 -> Rod 2) ---
connections_list = np.array(
    [[0, 1], [1, 2]]  # Cart connected to Bob 1  # Bob 1 connected to Bob 2
)

k_vals = np.array([k_val, k_val])

init_vecs = nodes_pos[connections_list[:, 0]] - nodes_pos[connections_list[:, 1]]
rest_lens = np.sqrt(np.sum(init_vecs**2, axis=1))

# --- 4. Masses & Damping ---
m_nodes = np.array([5.0, 1.0, 1.0])  # Cart is heavier than the pendulum bobs
m_diag = np.repeat(m_nodes, dims)
m_inv_diag = 1.0 / m_diag

c_nodes = np.array([c_val, c_val, c_val])
c_diag = np.repeat(c_nodes, dims)

# --- 5. Wall/Constraint Masks ---
# If you want the cart (Node 0) to ONLY move horizontally along X (y-axis locked to 0):
# You can use your wall_nodes logic or mask out Node 0's Y component.
wall_nodes = np.array(
    [0]
)  # Handled via custom U/mask if needed, or leave free to slide

# --- 6. External Input Matrix U (Driving the Cart with Hénon Map) ---
U = np.zeros((total_steps_with_free, matrix_size))

# Inject the Hénon map input directly into the Cart (Node 0, X-axis direction index 0)
target_node = 0
for i in range(min(len(henon_scaled), total_steps_with_free)):
    U[i, target_node * dims + 0] = henon_scaled[i, 0] * input_force

# --- 7. Run Your Pendulum Simulation ---
# (Using your custom run_pendulum_simulation function that includes gravity)
displacement, velocity = run_pendulum_simulation(
    steps=total_steps_with_free,
    dt=dt,
    m_inv_diag=m_inv_diag,
    c_diag=c_diag,
    gravity=9.81,
    initial_pos=nodes_pos,
    connections_list=connections_list,
    k_vals=k_vals,
    rest_lens=rest_lens,
    pivot_node=-1,  # No static ceiling pivot since Node 0 is a moving cart!
    U=U,
)

In [93]:
spring_animation(
    disp=displacement,
    nodes_pos=nodes_pos,
    connections_list=connections_list,
    highlighted_nodes=np.array([0, 1, 2]),
    external=True,
    is_3d=False,
    steps_jump=1,
).show()

In [ ]:
# --- 9. Downstream Reservoir Computing Evaluation (Your Existing Code) ---
X = np.column_stack((displacement, velocity))

X_delayed = X[:-tau_steps]
X_data = X_delayed[transient_steps_reservoir:]
Y_data = henon_scaled[transient_steps_reservoir + tau_steps :]

X_train, X_test = (
    X_data[:-test_steps],
    X_data[-test_steps:],
)
Y_train, Y_test = (
    Y_data[:-test_steps],
    Y_data[-test_steps:],
)

model = RidgeCV()
model.fit(X_train, Y_train)
Y_pred_scaled = model.predict(X_test)

Y_pred = henon_scaler.inverse_transform(Y_pred_scaled)
Y_test_orig = henon_scaler.inverse_transform(Y_test)
r_2 = r2_score(Y_test_orig, Y_pred)
mse = root_mean_squared_error(Y_test_orig, Y_pred)

print(f"Double Pendulum on Cart Reservoir -> R2: {r_2:.4f}, RMSE: {mse:.4f}")